In [1]:
"""
Symmetry-reduced (trivial-irrep / orbit-basis) Hamiltonian construction and
spectral + cyclicity diagnostics for a Rydberg/neutral-atom Ising model on a graph.

This is a curated / patched version of the original script. All flagged issues
have been addressed:

  (1) Robust igraph adjacency construction (mode="upper" on a 0/1 upper-triangular
      matrix) instead of the fragile / deprecated mode="undirected".
  (2) Graph counting no longer relies on os.listdir(); it uses the loader output
      length directly, so stray files (.DS_Store, README, caches) cannot break it.
  (3) Consistent `detuning` defaults across all entry points (no silent mismatch
      between the helper defaults and __main__).
  (4) Explicit float cast in the Rabi matrix element + an inline comment explaining
      that duplicate COO (j, k) entries are *intentional* and sum to the c_{jk}
      multiplicity on conversion to CSR.
  (5) Optional numerical Hermiticity assertion (verbose mode) as cheap insurance.
  (6) Spectrum degeneracy histogram reported in verbose mode so that a
      non-cyclic result can be attributed to degeneracy vs. a zero overlap.

Convention:
    H_G = sum_{(i,j) in E} V_int n_i n_j
          - detuning sum_i n_i
          + (omega / 2) sum_i sigma_i^x

Orbit basis:
    |O_k> = 1 / sqrt(|O_k|) * sum_{b in O_k} |b>
"""

import os

import numpy as np
import scipy.sparse as sp
import scipy.linalg as la
import igraph as ig
import networkx as nx
from numba import njit

from sym_graphs.utils.utils import SRG_loader

# Single source of truth for the default detuning, shared by every entry point
# so the helpers and __main__ can never silently disagree.
DEFAULT_DETUNING = 2.0 * np.pi


# ============================================================
# 1. Orbit basis construction
# ============================================================
@njit
def build_orbit_basis(N, generators):
    """
    Group the 2^N computational basis states into automorphism orbits.

    The orbit basis is:
        |O_k> = 1 / sqrt(|O_k|) * sum_{b in O_k} |b>

    Returns
    -------
    num_orbits : int
    orbit_id : np.ndarray, shape (2^N,)
        orbit_id[b] is the orbit index of bitstring b.
    orbit_sizes : np.ndarray, shape (num_orbits,)
    orbit_reprs : np.ndarray, shape (num_orbits,)

    Notes
    -----
    Generators are applied forward only (never inverted). This is sufficient:
    each permutation has finite order, so g^{-1} = g^{ord-1} is reached by
    repeated BFS application, and products of distinct generators are reached
    transitively. The semigroup orbit therefore equals the full group orbit.
    """
    dim = 1 << N
    num_gens = generators.shape[0]

    orbit_id = np.full(dim, -1, dtype=np.int32)
    orbit_sizes = np.zeros(dim, dtype=np.int32)
    orbit_reprs = np.zeros(dim, dtype=np.int32)
    queue = np.empty(dim, dtype=np.int32)

    num_orbits = 0
    for i in range(dim):
        if orbit_id[i] == -1:
            head = 0
            tail = 0
            queue[tail] = i
            tail += 1
            orbit_id[i] = num_orbits
            while head < tail:
                curr = queue[head]
                head += 1
                for g in range(num_gens):
                    nxt = 0
                    for bit in range(N):
                        if (curr >> bit) & 1:
                            nxt |= (1 << generators[g, bit])
                    if orbit_id[nxt] == -1:
                        orbit_id[nxt] = num_orbits
                        queue[tail] = nxt
                        tail += 1
            orbit_sizes[num_orbits] = tail
            orbit_reprs[num_orbits] = i
            num_orbits += 1

    return num_orbits, orbit_id, orbit_sizes[:num_orbits], orbit_reprs[:num_orbits]


# ============================================================
# 2. Trivial-sector Hamiltonian construction
# ============================================================
@njit
def build_trivial_sector_sparse(
    N,
    num_orbits,
    orbit_id,
    orbit_sizes,
    orbit_reprs,
    edges,
    omega,
    detuning,
    V_int,
):
    """
    Build the Hamiltonian block in the trivial symmetry sector.

    Convention used here:
        H_G = sum_{(i,j) in E} V_int n_i n_j
              - detuning sum_i n_i
              + (omega / 2) sum_i sigma_i^x

    If your paper convention is Omega * sigma_x, replace omega / 2.0 by omega
    in the off-diagonal term.
    """
    max_nnz = num_orbits * (N + 1)
    rows = np.empty(max_nnz, dtype=np.int32)
    cols = np.empty(max_nnz, dtype=np.int32)
    vals = np.empty(max_nnz, dtype=np.float64)
    idx = 0

    for k in range(num_orbits):
        x = orbit_reprs[k]

        # Diagonal terms (constant within an orbit, so evaluated on the rep).
        n_ones = 0
        for b in range(N):
            if (x >> b) & 1:
                n_ones += 1
        diag_val = -detuning * n_ones
        for e in range(edges.shape[0]):
            u = edges[e, 0]
            v = edges[e, 1]
            if ((x >> u) & 1) and ((x >> v) & 1):
                diag_val += V_int
        rows[idx] = k
        cols[idx] = k
        vals[idx] = diag_val
        idx += 1

        # Off-diagonal Rabi terms.
        #
        # NOTE: We emit one entry of (omega/2) * sqrt(|O_k| / |O_j|) PER bit-flip
        # that lands in orbit O_j. These duplicate (j, k) entries are INTENTIONAL:
        # scipy's COO -> CSR conversion sums duplicates, yielding the correct
        # matrix element (omega/2) * c_{jk} * sqrt(|O_k| / |O_j|), where c_{jk}
        # is the number of single-bit flips of the representative landing in O_j.
        # Hermiticity follows from c_{jk} |O_k| = c_{kj} |O_j|.
        for b in range(N):
            nxt = x ^ (1 << b)
            j = orbit_id[nxt]
            # Explicit float cast documents intent (orbit_sizes is int32).
            val = (omega / 2.0) * np.sqrt(
                np.float64(orbit_sizes[k]) / np.float64(orbit_sizes[j])
            )
            rows[idx] = j
            cols[idx] = k
            vals[idx] = val
            idx += 1

    return rows[:idx], cols[:idx], vals[:idx]


def get_generators_automatically(adj_matrix):
    """
    Compute automorphism generators using igraph/BLISS.

    Uses mode="upper" on a strict 0/1 upper-triangular matrix. This avoids the
    fragile / deprecated mode="undirected" path, which on a full symmetric
    matrix could create doubled edges or emit warnings depending on the
    python-igraph version.
    """
    N = adj_matrix.shape[0]

    # Strict upper triangle, booleanized, so each undirected edge is listed once.
    upper = np.triu((adj_matrix > 0).astype(np.int64), k=1)
    graph = ig.Graph.Adjacency(upper.tolist(), mode="upper")

    generators = graph.automorphism_group()
    generators = np.asarray(generators, dtype=np.int32)

    if generators.size == 0:
        # Trivial automorphism group -> every basis state is its own orbit.
        generators = np.empty((0, N), dtype=np.int32)

    return generators


def get_exact_trivial_sector(
    adj_matrix,
    generators,
    omega,
    detuning,
    V_int=1.0,
    verbose=False,
):
    """
    Build the exact Hamiltonian block in the trivial symmetry sector.

    Returns
    -------
    H_sparse : scipy.sparse.csr_matrix
    orbit_data : dict
        Contains num_orbits, orbit_id, orbit_sizes, orbit_reprs,
        zero_orbit_idx, and one_orbit_idx.
    """
    N = adj_matrix.shape[0]
    edges = np.argwhere(np.triu(adj_matrix, k=1)).astype(np.int32)
    generators = np.asarray(generators, dtype=np.int32)

    if verbose:
        print(
            f"Finding symmetry orbits for 2^{N} states "
            f"using {generators.shape[0]} generators..."
        )

    num_orbits, orbit_id, orbit_sizes, orbit_reprs = build_orbit_basis(
        N,
        generators,
    )

    zero_state = 0
    one_state = (1 << N) - 1
    zero_orbit_idx = int(orbit_id[zero_state])
    one_orbit_idx = int(orbit_id[one_state])

    if verbose:
        print(
            f"Graph symmetries collapsed the {1 << N} states "
            f"into exactly {num_orbits} orbits."
        )
        print(f"Orbit index of |0...0>: {zero_orbit_idx}")
        print(f"Size of |0...0> orbit: {orbit_sizes[zero_orbit_idx]}")
        print(f"Orbit index of |1...1>: {one_orbit_idx}")
        print(f"Size of |1...1> orbit: {orbit_sizes[one_orbit_idx]}")
        print("Building exact block matrix in sparse format...")

    rows, cols, vals = build_trivial_sector_sparse(
        N,
        num_orbits,
        orbit_id,
        orbit_sizes,
        orbit_reprs,
        edges,
        omega,
        detuning,
        V_int,
    )

    H_sparse = sp.coo_matrix(
        (vals, (rows, cols)),
        shape=(num_orbits, num_orbits),
    ).tocsr()

    if verbose:
        # Cheap insurance: the orbit-basis block must be symmetric (real H).
        asym = abs(H_sparse - H_sparse.T)
        max_asym = 0.0 if asym.nnz == 0 else float(asym.max())
        print(f"Hermiticity check max|H - H^T|: {max_asym:.3e}")
        assert max_asym < 1e-9, "Trivial-sector Hamiltonian is not symmetric!"

    orbit_data = {
        "num_orbits": num_orbits,
        "orbit_id": orbit_id,
        "orbit_sizes": orbit_sizes,
        "orbit_reprs": orbit_reprs,
        "zero_orbit_idx": zero_orbit_idx,
        "one_orbit_idx": one_orbit_idx,
    }

    return H_sparse, orbit_data


def get_exact_srg_trivial_sector(
    adj_matrix,
    generators,
    omega,
    detuning,
    V_int=1.0,
    verbose=False,
):
    """
    Backward-compatible alias for the old function name.
    """
    return get_exact_trivial_sector(
        adj_matrix=adj_matrix,
        generators=generators,
        omega=omega,
        detuning=detuning,
        V_int=V_int,
        verbose=verbose,
    )


# ============================================================
# 3. Initial states in the orbit basis
# ============================================================
def build_initial_state_in_orbit_basis(N, orbit_data, initialization="zero"):
    """
    Build the initial state vector in the normalized orbit basis.

    Available choices
    -----------------
    initialization = "zero"  ->  |psi_0> = |0...0>
    initialization = "one"   ->  |psi_0> = |1...1>
    initialization = "plus"  ->  |psi_0> = |+>^{tensor N} = 1/sqrt(2^N) sum_b |b>

    In the normalized orbit basis the plus state has coordinates:
        psi_k = sqrt(|O_k| / 2^N).
    """
    orbit_id = orbit_data["orbit_id"]
    orbit_sizes = orbit_data["orbit_sizes"]
    num_orbits = orbit_data["num_orbits"]

    psi_init = np.zeros(num_orbits, dtype=np.float64)

    if initialization == "zero":
        zero_state = 0
        zero_orbit_idx = int(orbit_id[zero_state])
        psi_init[zero_orbit_idx] = 1.0
    elif initialization == "one":
        one_state = (1 << N) - 1
        one_orbit_idx = int(orbit_id[one_state])
        psi_init[one_orbit_idx] = 1.0
    elif initialization == "plus":
        psi_init = np.sqrt(orbit_sizes.astype(np.float64) / float(1 << N))
        # Numerical safety (already normalized since sum |O_k| = 2^N).
        psi_init = psi_init / np.linalg.norm(psi_init)
    else:
        raise ValueError(
            "Unknown initialization. Use initialization='zero', 'one', or 'plus'."
        )

    return psi_init


# ============================================================
# 4. Spectral and cyclicity diagnostics
# ============================================================
def analyze_trivial_sector_cyclicity(
    adj_matrix,
    graph_name,
    omega,
    detuning,
    V_int=1.0,
    gap_tolerance=1e-10,
    overlap_tolerance=1e-12,
    initialization="zero",
    verbose=False,
):
    """
    Compute spectral and cyclicity diagnostics in the trivial sector.

    For a chosen initial state psi_0, cyclicity in H_0 is guaranteed by:
        1. H_0 has simple spectrum;
        2. |<phi_l | psi_0>|^2 > 0 for every eigenvector phi_l.

    Numerically:
        min_gap > gap_tolerance
        and
        min_squared_overlap > overlap_tolerance.
    """
    N = adj_matrix.shape[0]
    generators = get_generators_automatically(adj_matrix)

    H_sparse, orbit_data = get_exact_trivial_sector(
        adj_matrix=adj_matrix,
        generators=generators,
        omega=omega,
        detuning=detuning,
        V_int=V_int,
        verbose=verbose,
    )

    H_dense = H_sparse.toarray()
    eigenvalues, eigenvectors = la.eigh(H_dense)

    spectrum_size = len(eigenvalues)
    if spectrum_size > 1:
        gaps = np.diff(eigenvalues)
        min_gap = float(np.min(gaps))
    else:
        gaps = np.array([], dtype=np.float64)
        min_gap = np.inf

    is_non_degenerate = bool(min_gap > gap_tolerance)

    # Degeneracy multiplicity (number of eigenvalues clustered within tol).
    # Lets a non-cyclic verdict be attributed to degeneracy vs. zero overlap.
    num_degenerate_pairs = int(np.sum(gaps <= gap_tolerance))

    psi_init = build_initial_state_in_orbit_basis(
        N=N,
        orbit_data=orbit_data,
        initialization=initialization,
    )
    psi_norm = float(np.linalg.norm(psi_init))

    # scipy.linalg.eigh returns eigenvectors as columns:
    #   eigenvectors[:, ell] = |phi_ell>
    # Therefore <phi_ell | psi_init> = eigenvectors[:, ell]^dagger @ psi_init.
    squared_overlaps = np.abs(eigenvectors.conj().T @ psi_init) ** 2
    min_squared_overlap = float(np.min(squared_overlaps))
    max_squared_overlap = float(np.max(squared_overlaps))
    sum_squared_overlaps = float(np.sum(squared_overlaps))
    zero_overlap_count = int(np.sum(squared_overlaps <= overlap_tolerance))
    support_dimension = int(np.sum(squared_overlaps > overlap_tolerance))
    has_full_support = bool(min_squared_overlap > overlap_tolerance)

    is_cyclic = bool(is_non_degenerate and has_full_support)

    if verbose:
        print(
            f"Degenerate gap pairs (<= {gap_tolerance:.1e}): "
            f"{num_degenerate_pairs}"
        )

    result = {
        "graph": graph_name,
        "initialization": initialization,
        "graph_size": int(N),
        "num_generators": int(generators.shape[0]),
        "spectrum_size": int(spectrum_size),
        "zero_orbit_idx": int(orbit_data["zero_orbit_idx"]),
        "zero_orbit_size": int(orbit_data["orbit_sizes"][orbit_data["zero_orbit_idx"]]),
        "one_orbit_idx": int(orbit_data["one_orbit_idx"]),
        "one_orbit_size": int(orbit_data["orbit_sizes"][orbit_data["one_orbit_idx"]]),
        "psi_norm": psi_norm,
        "min_spectral_gap": min_gap,
        "is_non_degenerate": is_non_degenerate,
        "num_degenerate_pairs": num_degenerate_pairs,
        "min_squared_overlap": min_squared_overlap,
        "max_squared_overlap": max_squared_overlap,
        "sum_squared_overlaps": sum_squared_overlaps,
        "zero_overlap_count": zero_overlap_count,
        "support_dimension": support_dimension,
        "has_full_support": has_full_support,
        "is_cyclic": is_cyclic,
        "gap_tolerance": gap_tolerance,
        "overlap_tolerance": overlap_tolerance,
    }

    return result


def print_cyclicity_result(result):
    """
    Pretty-print one result dictionary.
    """
    print(result["graph"])
    print(f"Initialization: {result['initialization']}")
    print(f"Graph size: {result['graph_size']}")
    print(f"Number of automorphism generators: {result['num_generators']}")
    print(f"Spectrum size / dim(H_0): {result['spectrum_size']}")
    print(f"Zero-state orbit index: {result['zero_orbit_idx']}")
    print(f"Zero-state orbit size: {result['zero_orbit_size']}")
    print(f"One-state orbit index: {result['one_orbit_idx']}")
    print(f"One-state orbit size: {result['one_orbit_size']}")
    print(f"Initial-state norm: {result['psi_norm']:.12f}")
    print(f"Minimum spectral gap: {result['min_spectral_gap']:.4e}")
    print(
        "Is the trivial-sector spectrum non-degenerate? "
        f"{result['is_non_degenerate']}"
    )
    print(f"Degenerate gap pairs: {result['num_degenerate_pairs']}")
    print(f"Minimum squared overlap: {result['min_squared_overlap']:.4e}")
    print(f"Maximum squared overlap: {result['max_squared_overlap']:.4e}")
    print(f"Sum of squared overlaps: {result['sum_squared_overlaps']:.12f}")
    print(
        f"Number of eigenvectors with squared overlap <= "
        f"{result['overlap_tolerance']:.1e}: {result['zero_overlap_count']}"
    )
    print(f"Spectral support dimension: {result['support_dimension']}")
    print(
        f"Does the initial state have full spectral support? "
        f"{result['has_full_support']}"
    )
    print(f"Is the initial state cyclic in H_0? {result['is_cyclic']}")
    print("")


def print_summary_table(results):
    """
    Print compact terminal table.
    """
    header = (
        f"{'Graph':40s} "
        f"{'init':>6s} "
        f"{'N':>4s} "
        f"{'dim(H0)':>10s} "
        f"{'support':>9s} "
        f"{'min_gap':>14s} "
        f"{'min_ovlp^2':>14s} "
        f"{'#zero_ovlp':>11s} "
        f"{'cyclic':>8s}"
    )
    print(header)
    print("-" * len(header))
    for r in results:
        print(
            f"{r['graph']:40s} "
            f"{r['initialization']:>6s} "
            f"{r['graph_size']:4d} "
            f"{r['spectrum_size']:10d} "
            f"{r['support_dimension']:9d} "
            f"{r['min_spectral_gap']:14.4e} "
            f"{r['min_squared_overlap']:14.4e} "
            f"{r['zero_overlap_count']:11d} "
            f"{str(r['is_cyclic']):>8s}"
        )


def results_to_latex_rows(results):
    """
    Generate LaTeX table rows.

    Columns:
        Graph, Initialization, Graph size, dim(H_0), support dimension,
        min spectral gap, min squared overlap, cyclic?
    """
    rows = []
    for r in results:
        cyclic_str = "Yes" if r["is_cyclic"] else "No"
        row = (
            f"{r['graph']} & "
            f"{r['initialization']} & "
            f"{r['graph_size']} & "
            f"{r['spectrum_size']} & "
            f"{r['support_dimension']} & "
            f"${r['min_spectral_gap']:.2e}$ & "
            f"${r['min_squared_overlap']:.2e}$ & "
            f"{cyclic_str} \\\\"
        )
        rows.append(row)
    return "\n".join(rows)


# ============================================================
# 5. SRG analysis
# ============================================================
def run_srg_analysis(
    dataset_root="../dataset/SRG",
    sizes=(5, 9, 10, 13, 15, 16, 17, 21),
    omega=np.pi,
    detuning=DEFAULT_DETUNING,
    V_int=1.0,
    gap_tolerance=1e-10,
    overlap_tolerance=1e-12,
    initialization="zero",
    verbose=False,
):
    """
    Run the spectral + cyclicity analysis on the SRG dataset.
    """
    sizes = list(sizes)
    Graphs = SRG_loader(dataset_root, sizes)

    results = []
    print("--- Strongly Regular Graphs ---")
    print(f"Initialization: {initialization}")

    for s in sizes:
        # Use the loader output length directly. This is robust to stray files
        # in the dataset directory (.DS_Store, README, caches, etc.).
        graphs_for_size = Graphs[s]
        n_s = len(graphs_for_size)
        for idx in range(n_s):
            adj_matrix = graphs_for_size[idx]
            graph_name = f"SRG_N{s}_idx{idx}"
            result = analyze_trivial_sector_cyclicity(
                adj_matrix=adj_matrix,
                graph_name=graph_name,
                omega=omega,
                detuning=detuning,
                V_int=V_int,
                gap_tolerance=gap_tolerance,
                overlap_tolerance=overlap_tolerance,
                initialization=initialization,
                verbose=verbose,
            )
            print_cyclicity_result(result)
            results.append(result)

    return results


# ============================================================
# 6. Highly symmetric graph analysis
# ============================================================
def make_highly_symmetric_graphs():
    """
    Build the named highly symmetric graphs.
    """
    clebsch_edge_list = [
        (7, 10), (10, 11), (5, 8), (5, 7), (5, 13),
        (7, 15), (0, 1), (0, 10), (0, 6), (0, 8),
        (0, 13), (1, 3), (1, 4), (1, 9), (1, 15),
        (2, 3), (2, 10), (2, 8), (2, 12), (2, 15),
        (3, 6), (3, 5), (3, 11), (4, 10), (4, 5),
        (4, 12), (4, 14), (6, 7), (6, 12), (6, 14),
        (8, 9), (8, 14), (7, 9), (9, 11), (9, 12),
        (11, 13), (11, 14), (12, 13), (13, 15), (14, 15),
    ]
    Clebsch = nx.Graph()
    Clebsch.add_edges_from(clebsch_edge_list)

    graphs_dict = {
        "Petersen graph": nx.petersen_graph(),
        "Icosahedral graph": nx.icosahedral_graph(),
        "Heawood graph": nx.heawood_graph(),
        "Clebsch graph": Clebsch,
        "Pappus graph": nx.pappus_graph(),
        "Desargues graph": nx.desargues_graph(),
        "Dodecahedral graph": nx.dodecahedral_graph(),
    }
    return graphs_dict


def run_highly_symmetric_analysis(
    omega=np.pi,
    detuning=DEFAULT_DETUNING,
    V_int=1.0,
    gap_tolerance=1e-10,
    overlap_tolerance=1e-12,
    initialization="zero",
    verbose=False,
):
    """
    Run the spectral + cyclicity analysis on the named graphs.
    """
    graphs_dict = make_highly_symmetric_graphs()

    results = []
    print("--- Highly Symmetric Graphs ---")
    print(f"Initialization: {initialization}")

    for name, G in graphs_dict.items():
        adj_matrix = nx.to_numpy_array(G, dtype=np.float64)
        result = analyze_trivial_sector_cyclicity(
            adj_matrix=adj_matrix,
            graph_name=name,
            omega=omega,
            detuning=detuning,
            V_int=V_int,
            gap_tolerance=gap_tolerance,
            overlap_tolerance=overlap_tolerance,
            initialization=initialization,
            verbose=verbose,
        )
        print_cyclicity_result(result)
        results.append(result)

    return results


# ============================================================
# 7. Main execution
# ============================================================
if __name__ == "__main__":
    omega = np.pi
    detuning = DEFAULT_DETUNING
    V_int = 1.0
    gap_tolerance = 1e-12
    overlap_tolerance = 1e-12

    srg_sizes = [5, 9, 10, 13, 15, 16, 17, 21]

    # Choose one of:
    #     "zero" -> |0...0>
    #     "one"  -> |1...1>
    #     "plus" -> |+...+>
    #
    # Or set run_all_initializations = True below to run all three.
    initialization = "zero"
    run_all_initializations = False

    all_results = []

    if run_all_initializations:
        initializations = ["zero", "one", "plus"]
    else:
        initializations = [initialization]

    for init in initializations:
        print("")
        print("====================================================")
        print(f"Running initialization: {init}")
        print("====================================================")
        print("")

        srg_results = run_srg_analysis(
            dataset_root="../dataset/SRG",
            sizes=srg_sizes,
            omega=omega,
            detuning=detuning,
            V_int=V_int,
            gap_tolerance=gap_tolerance,
            overlap_tolerance=overlap_tolerance,
            initialization=init,
            verbose=False,
        )

        symmetric_results = run_highly_symmetric_analysis(
            omega=omega,
            detuning=detuning,
            V_int=V_int,
            gap_tolerance=gap_tolerance,
            overlap_tolerance=overlap_tolerance,
            initialization=init,
            verbose=False,
        )

        all_results.extend(srg_results + symmetric_results)

    print("")
    print("=== Compact Summary Table ===")
    print_summary_table(all_results)

    print("")
    print("=== LaTeX Rows ===")
    print(results_to_latex_rows(all_results))



Running initialization: zero

--- Strongly Regular Graphs ---
Initialization: zero
SRG_N5_idx0
Initialization: zero
Graph size: 5
Number of automorphism generators: 2
Spectrum size / dim(H_0): 8
Zero-state orbit index: 0
Zero-state orbit size: 1
One-state orbit index: 7
One-state orbit size: 1
Initial-state norm: 1.000000000000
Minimum spectral gap: 6.2667e-01
Is the trivial-sector spectrum non-degenerate? True
Degenerate gap pairs: 0
Minimum squared overlap: 3.1071e-06
Maximum squared overlap: 7.5740e-01
Sum of squared overlaps: 1.000000000000
Number of eigenvectors with squared overlap <= 1.0e-12: 0
Spectral support dimension: 8
Does the initial state have full spectral support? True
Is the initial state cyclic in H_0? True

SRG_N9_idx0
Initialization: zero
Graph size: 9
Number of automorphism generators: 3
Spectrum size / dim(H_0): 26
Zero-state orbit index: 0
Zero-state orbit size: 1
One-state orbit index: 25
One-state orbit size: 1
Initial-state norm: 1.000000000000
Minimum spect